# HydraY NNUE - bucket 0: 1,18B + 50M di finali

Runtime → Cambia tipo di runtime → **GPU (T4)**, poi Runtime → **Esegui tutte**.
Durata ~1h. **Non lasciare la scheda inattiva.**

### A cosa serve
Il bucket di uscita 0 (2-5 pezzi) e' lo **0,23%** del dataset attuale, contro il
12,5% che avrebbe una distribuzione uniforme: sotto-rappresentato di un fattore
57. La conseguenza si misura: la rete 3.0.0 valuta **donna contro re nudo 128
cp** invece dei ~900 che vale davvero. In partita lo maschera il probe delle
tablebase, ma appena esci dal loro raggio la rete non sa di stare vincendo.

Qui aggiungiamo **50.734.150 posizioni di soli finali** (86% bucket 0, 14%
bucket 1) generate con l'etichettatore 3.0.0 e adjudication Syzygy attiva.
Bucket 0 passa da 0,23% a ~3,8%: un fattore 17.

### Perche' il prefisso da 1,18B e non i 2,75B
A5 (2026-08-02) ha misurato che i 1,57B in piu' non valgono niente: −7,30 ±8,61.
Pagarli di nuovo costerebbe 2,5x il tempo per lo stesso risultato. Il prefisso da
1,18B e' anche **esattamente** il set su cui e' addestrata la 3.0.0, quindi il
confronto finale ha una sola variabile: il batch di finali.

E il file sta in ~39 GB, quindi entra nel disco Colab in un colpo solo: **un
unico run**, niente training a tappe, niente tappe da sbagliare.

### Il controllo che rende questo esperimento diverso
Alla fine il notebook stampa i sanity eval. **KQvK deve salire da 128 cp verso
qualcosa di sensato.** Se non succede, l'esperimento e' fallito e lo sai subito,
senza spendere 4000 partite di SPRT per scoprirlo.


In [ ]:
# --- helper: qualunque comando fallito ferma il notebook, e l'output si vede ---
import subprocess, os, sys, json

def sh(cmd):
    # L'output va riletto e ristampato da Python: subprocess.run() senza capture
    # scrive sui file descriptor del KERNEL, che Colab non mostra nella cella.
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash',
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait() != 0:
        raise RuntimeError(f'FALLITO (exit {p.returncode}): {cmd}')

sh('nvidia-smi --query-gpu=name,memory.total --format=csv')
sh('df -h /content | tail -1')
print('\nGPU presente. Se la riga sopra non mostra una T4, cambia runtime.')

In [ ]:
# --- Drive + dataset ---
from google.colab import drive
drive.mount('/content/drive')

import glob
cand = glob.glob('/content/drive/MyDrive/**/hydray_v5_1180M_eg50M_shuffled.bin.zst', recursive=True)
assert cand, 'dataset non trovato su Drive'
SRC = cand[0]
os.environ['SRC'] = SRC
print('trovato:', SRC, os.path.getsize(SRC), 'byte')

NET_ID   = 'hydray-eg50m'
TOTAL_SB = 40
TRAINER  = '/content/th/nnue/trainer'

In [ ]:
# --- Rust ---
sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
sh('$HOME/.cargo/bin/cargo --version')

In [ ]:
# --- clone + verifica architettura ---
# La verifica non e' cerimoniale: un run precedente ha addestrato per un'ora
# un'architettura diversa da quella creduta, perche' il clone era sbagliato e il
# nome del checkpoint non dice nulla sul contenuto.
sh('rm -rf /content/th')
sh('git clone --depth 1 --branch dev https://github.com/ThomasGhione/HydraY /content/th')

src = open(f'{TRAINER}/src/bin/sanity.rs').read()
assert 'const HIDDEN: usize = 512;' in src, 'atteso lo strato nascosto a 512 (la 1024 ha perso)'
assert 'const INPUT_BUCKETS: usize = 4;' in src, 'attesi 4 king bucket'
print('branch dev, 512 neuroni, 4 king bucket: ok')

In [ ]:
# --- decompressione locale (il mount Drive cachea tutto cio' che legge:
# zstd legge una volta sola e scrive su disco locale, poi la cache si libera) ---
sh('apt-get -qq install -y zstd >/dev/null')

# Picco = .zst tenuto in cache dal mount + file scompattato sul disco locale.
# La cache viene rilasciata solo quando zstd esce, quindi convivono.
DECOMP = 39383496384
need_gb = (os.path.getsize(SRC) + DECOMP) / 2**30
st = os.statvfs('/content')
free_gb = st.f_bavail * st.f_frsize / 2**30
print(f'servono ~{need_gb:.1f} GiB di picco, liberi {free_gb:.1f} GiB')
assert free_gb > need_gb + 1.5, 'disco insufficiente'

sh('zstd -d -T0 --long=27 "$SRC" -o /content/data.bin')
SIZE = os.path.getsize('/content/data.bin')
print('data.bin:', SIZE, 'byte =', SIZE // 32, 'posizioni')
assert SIZE == DECOMP, f'taglia inattesa: {SIZE}'
sh('df -h /content | tail -1')

In [ ]:
# --- validation set: 1000 MiB dalla coda, mai addestrati ---
TEST_MIB = 1000
skip_mib = SIZE // (1024*1024) - TEST_MIB
sh(f'dd if=/content/data.bin bs=1M skip={skip_mib} count={TEST_MIB} of=/content/test.bin status=progress')
print('test.bin:', os.path.getsize('/content/test.bin'), 'byte')

In [ ]:
# --- training: un solo run, 40 superbatch ---
sh(f'cd {TRAINER} && PATH=$HOME/.cargo/bin:$PATH CUDA_PATH=/usr/local/cuda '
   f'TEST_PATH=/content/test.bin '
   f'cargo run -r --bin trainer --features cuda -- '
   f'/content/data.bin {TOTAL_SB} {NET_ID}')

In [ ]:
# --- verifica finale e salvataggio su Drive ---
final = f'{TRAINER}/checkpoints/{NET_ID}-{TOTAL_SB}/quantised.bin'
sz = os.path.getsize(final)
assert sz == 3163200, f'taglia {sz}: attesa 3.163.200 (512 neuroni, 4 king bucket)'
print('quantised.bin:', sz, 'byte - architettura confermata\n')

sh(f'cd {TRAINER} && PATH=$HOME/.cargo/bin:$PATH cargo run -r --bin sanity -- {final}')
sh(f'cp -r {TRAINER}/checkpoints/{NET_ID}-{TOTAL_SB} /content/drive/MyDrive/')
print('\n' + '='*60)
print('GUARDA LA RIGA "KQvK" QUI SOPRA.')
print('  3.0.0 dava 128 cp. Il valore vero e ~900 o direttamente matto.')
print('  Se e salita di poco, il batch di finali non ha funzionato.')
print('='*60)

## Come leggere il risultato

**Primo controllo, immediato: KQvK.** La 3.0.0 dava 128 cp. Se il batch di
finali ha funzionato quel numero deve salire in modo netto. Se resta sotto i 200,
l'esperimento e' fallito e non vale la pena spendere lo SPRT.

**Secondo: gli altri sanity eval non devono peggiorare.** startpos ~+50,
mediogioco ~900, donna in piu' ~1700. Il feature transformer `l0` e' condiviso
fra tutti i bucket, quindi il 4% di posizioni di finale in piu' potrebbe in
teoria spostare la rappresentazione a scapito del mediogioco. Se quei numeri
crollano, il prezzo e' troppo alto.

**Terzo: la validation loss.** Non e' confrontabile con quella dei run
precedenti - il test set e' diverso, perche' contiene anche finali.

Poi tocca allo SPRT testa a testa contro la rete della 3.0.0.
